In [1]:
import sys
from ortools.sat.python import cp_model

In [2]:
def read_input():
    data = list(map(int, sys.stdin.read().split()))
    n, m = data[0], data[1]
    idx = 2
    edges = []
    for _ in range(m):
        edges.append((data[idx] - 1, data[idx + 1] - 1))
        idx += 2

    t, c = [[0] * n for _ in range(n)], [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            t[i][j], c[i][j] = data[idx], data[idx + 1]
            idx += 2
    
    L = data[idx]
    return n, m, edges, t, c, L

In [5]:
def solve_ortools(n, m, edges, t, c, L):
    x = {}
    model = cp_model.CpModel()
    solver = cp_model.CpSolver()
    y = [model.new_int_var(0, L, f"y[{i}]") for i in range(n)]
    model.add(y[0] == 0)

    M = 10**7
    adj = [[] for _ in range(n)]
    cost_lst = []
    for (i, j) in edges:
        x[(i, j)] = model.new_bool_var(f"x[{i}][{j}]")
        x[(j, i)] = model.new_bool_var(f"x[{j}][{i}]")
        
        model.add(y[i] + t[i][j] - (1 - x[(i, j)]) * M <= y[j])
        model.add(y[i] + t[i][j] + (1 - x[(i, j)]) * M >= y[j])

        model.add(y[j] + t[j][i] - (1 - x[(j, i)]) * M <= y[i])
        model.add(y[j] + t[j][i] + (1 - x[(j, i)]) * M >= y[i])
        
        adj[i].append(j)
        adj[j].append(i)
        cost_lst.append(x[(i, j)] * c[i][j])
        cost_lst.append(x[(j, i)] * c[j][i])

    for j in range(n):
        if len(adj[j]) != 0:
            model.add(sum(x[(i, j)] for i in adj[j]) == 1)

    model.minimize(sum(cost_lst))
    status = solver.solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(solver.ObjectiveValue())